# 부산 → 인천공항 노선 구간 분할
## 목적
카카오모빌리티 길찾기 API로 부산 물류센터 → 인천공항 경로를 받아와서,
휴게소 단위로 구간을 분할하고 각 구간의 거리·평균속도를 계산한다.

이 결과는 이후 ML 회귀 모델(kWh/100km 예측)의 입력값(X)으로 사용된다.

**사전 준비**
1. https://developers.kakao.com 가입 → 애플리케이션 생성 → REST API 키 발급
2. 아래 `REST_API_KEY`에 발급받은 키 입력


In [16]:
import os
from dotenv import load_dotenv
import requests
import pandas as pd
import numpy as np

load_dotenv()
REST_API_KEY = os.getenv("KAKAO_REST_API_KEY")
HEADERS = {"Authorization": f"KakaoAK {REST_API_KEY}"}

print(f"키 로드됨: {REST_API_KEY[:6]}****" if REST_API_KEY else "키를 못 찾았습니다")

키 로드됨: ed6f28****


## 1. 출발지/도착지 좌표 설정

카카오 API는 좌표를 **경도,위도** 순서로 받는다 (위도,경도 아님 — 헷갈리기 쉬우니 주의).

좌표를 모르면 카카오 로컬 API(주소 검색)로 변환하거나, 카카오맵에서 우클릭 → 좌표 복사로 얻을 수 있다.


In [17]:
# def search_place(keyword):
#     url = "https://dapi.kakao.com/v2/local/search/keyword.json"
#     params = {"query": keyword}
#     res = requests.get(url, headers=HEADERS, params=params)
#     res.raise_for_status()
#     docs = res.json()["documents"]
#     return docs[0] if docs else None
# # 이 이름과 관련된 장소를 찾는 방식
# origin_result = search_place("부산항 신항")
# print(origin_result)

In [18]:
def search_address(address):
    url = "https://dapi.kakao.com/v2/local/search/address.json"
    params = {"query": address}
    res = requests.get(url, headers=HEADERS, params=params)
    res.raise_for_status()
    docs = res.json()["documents"]
    return docs[0] if docs else None

# 부산 실제 주소로 검색 (프로젝트에서 정한 부산 물류센터 주소로 바꿔도 됨)
origin_result = search_address("부산광역시 강서구 녹산산단232로 38-26")
print(origin_result)

{'address': {'address_name': '부산 강서구 송정동 1709-2', 'b_code': '2644010900', 'h_code': '2644056000', 'main_address_no': '1709', 'mountain_yn': 'N', 'region_1depth_name': '부산', 'region_2depth_name': '강서구', 'region_3depth_h_name': '녹산동', 'region_3depth_name': '송정동', 'sub_address_no': '2', 'x': '128.840263337127', 'y': '35.0865743070385'}, 'address_name': '부산 강서구 녹산산단232로 38-26', 'address_type': 'ROAD_ADDR', 'road_address': {'address_name': '부산 강서구 녹산산단232로 38-26', 'building_name': '부산진해경제자유구역청', 'main_building_no': '38', 'region_1depth_name': '부산', 'region_2depth_name': '강서구', 'region_3depth_name': '송정동', 'road_name': '녹산산단232로', 'sub_building_no': '26', 'underground_yn': 'N', 'x': '128.840263337127', 'y': '35.0865743070385', 'zone_no': '46757'}, 'x': '128.840263337127', 'y': '35.0865743070385'}


In [19]:
import requests

r = requests.get(

    "https://dapi.kakao.com/v2/local/search/keyword.json",

    headers={"Authorization": f"KakaoAK {REST_API_KEY}"},

    params={"query": "인천공항 제1여객터미널"},

)

doc = r.json()["documents"][0]

DESTINATION = f"{doc['x']},{doc['y']}"   # x=경도, y=위도

print(doc["place_name"], DESTINATION)

인천국제공항 제1여객터미널 126.45240378314084,37.449691917253595


In [20]:
print("ORIGIN =", ORIGIN)
import json
print(json.dumps(route_json, indent=2, ensure_ascii=False))

ORIGIN = 128.840263337127,35.0865743070385
{
  "trans_id": "01a045da2ee175fbaf63f51d49a97fa0",
  "routes": [
    {
      "result_code": 103,
      "result_msg": "도착 지점 주변의 도로를 탐색할 수 없음"
    }
  ]
}


## 2. 카카오 길찾기 API 호출

`priority`를 `RECOMMEND`(추천 경로) 또는 `HIGHWAY_UNIONTOLL`(고속도로 우선) 중 선택 가능.
물류 운송이므로 고속도로 고정 노선을 원하면 후자를 고려.


In [21]:
def get_route(origin, destination, priority="RECOMMEND"):
    url = "https://apis-navi.kakaomobility.com/v1/directions"
    params = {
        "origin": origin,
        "destination": destination,
        "priority": priority,
        "road_details": True
    }
    res = requests.get(url, headers=HEADERS, params=params)
    res.raise_for_status()
    return res.json()

route_json = get_route(ORIGIN, DESTINATION)

# 전체 요약 확인
summary = route_json["routes"][0]["summary"]
print(f"총 거리: {summary['distance']/1000:.1f} km")
print(f"예상 소요시간: {summary['duration']/60:.0f} 분")


총 거리: 423.4 km
예상 소요시간: 363 분


## 3. 세부 도로(roads) 단위로 추출

카카오 응답의 `roads` 배열은 도로명이 바뀔 때마다 잘게 쪼개져 있다 (수십~수백 개).
이 상태로는 ML 입력으로 쓰기엔 너무 세분화되어 있어, 다음 단계에서 휴게소 단위로 다시 묶는다.


In [22]:
def extract_road_segments(route_json):
    rows = []
    for section in route_json["routes"][0]["sections"]:
        for road in section["roads"]:
            rows.append({
                "도로명": road.get("name", "무명도로"),
                "거리_m": road["distance"],
                "소요시간_s": road["duration"],
                "vertexes": road["vertexes"]  # [lng1,lat1,lng2,lat2,...] 형태의 flat list
            })
    return pd.DataFrame(rows)

df_roads = extract_road_segments(route_json)
df_roads["평균속도_kmh"] = (df_roads["거리_m"]/1000) / (df_roads["소요시간_s"]/3600)
df_roads.head()


,도로명,거리_m,소요시간_s,vertexes,평균속도_kmh
0,,20,5,"[128.84042123086942, 35.086171234637426, 128.8...",14.40000
1,,25,6,"[128.8402666945559, 35.08607314909686, 128.840...",15.00000
2,,23,5,"[128.84026438362656, 35.085847809565, 128.8402...",16.56000
3,녹산산단232로,74,6,"[128.8402732232017, 35.08564042120672, 128.839...",44.40000
4,가락대로,238,31,"[128.83947291844902, 35.08566399241111, 128.83...",27.63871


## 4. 전체 경로 좌표(polyline) 하나로 합치기

각 도로의 vertexes를 순서대로 이어붙여 전체 경로 좌표 리스트를 만든다.
이걸 기준으로 휴게소 위치를 매칭한다.


In [23]:
def flatten_route_coords(df_roads):
    coords = []  # (lng, lat) 튜플 리스트, 누적거리 포함
    cum_dist = 0.0
    for _, row in df_roads.iterrows():
        v = row["vertexes"]
        pts = [(v[i], v[i+1]) for i in range(0, len(v), 2)]
        for j in range(len(pts) - 1):
            lng1, lat1 = pts[j]
            lng2, lat2 = pts[j+1]
            # 간단 유클리드 근사 (정밀 계산 필요시 haversine 사용)
            seg_len_deg = np.hypot(lng2 - lng1, lat2 - lat1)
            coords.append({"lng": lng1, "lat": lat1, "cum_dist_deg": cum_dist})
            cum_dist += seg_len_deg
    coords.append({"lng": pts[-1][0], "lat": pts[-1][1], "cum_dist_deg": cum_dist})
    return pd.DataFrame(coords)

route_coords = flatten_route_coords(df_roads)
# 도(degree) 단위 누적거리를 실제 거리(km) 스케일에 맞춰 보정
scale = df_roads["거리_m"].sum() / 1000 / route_coords["cum_dist_deg"].iloc[-1]
route_coords["cum_dist_km"] = route_coords["cum_dist_deg"] * scale
route_coords.head()


,lng,lat,cum_dist_deg,cum_dist_km
0,128.840421,35.086171,0.000000,0.000000
1,128.840421,35.086171,0.000000,0.000000
2,128.840388,35.086162,0.000034,0.003411
3,128.840377,35.086108,0.000089,0.008929
4,128.840344,35.086082,0.000132,0.013191


## 5. 노선상 주요 휴게소 좌표 매칭

부산→인천공항 경로상에 있는 실제 휴게소 목록 (경부고속도로 상행선 + 인천공항고속도로).

**TODO: 아래 좌표는 예시이므로, 카카오 로컬 API(주소/키워드 검색)로 정확한 좌표를 확인 후 교체할 것.**


In [28]:
def rest_areas_along_route(route_coords, step_km=25, radius=3000):
    found = {}
    marks = np.arange(step_km, route_coords["cum_dist_km"].iloc[-1], step_km)
    for i in np.searchsorted(route_coords["cum_dist_km"].values, marks):
        p = route_coords.iloc[i]
        docs = requests.get(
            "https://dapi.kakao.com/v2/local/search/keyword.json",
            headers=HEADERS,
            params={"query": "휴게소", "x": p["lng"], "y": p["lat"],
                    "radius": radius, "sort": "distance", "size": 5},
        ).json()["documents"]
        for d in docs:
            if "휴게소" in d["place_name"]:
                found[d["id"]] = {"name": d["place_name"], "lng": float(d["x"]),
                                  "lat": float(d["y"]), "cum_km": p["cum_dist_km"]}
    return sorted(found.values(), key=lambda r: r["cum_km"])

def search_place(keyword):
    """카카오 로컬 API로 장소명 -> 좌표 변환"""
    url = "https://dapi.kakao.com/v2/local/search/keyword.json"
    params = {"query": keyword}
    res = requests.get(url, headers=HEADERS, params=params)
    res.raise_for_status()
    docs = res.json()["documents"]
    if not docs:
        return None
    return {"lng": float(docs[0]["x"]), "lat": float(docs[0]["y"])}

for ra in rest_areas:
    coord = search_place(ra["name"])
    if coord:
        ra.update(coord)
    else:
        print(f"좌표 못찾음: {ra['name']}")

df_rest = pd.DataFrame(rest_areas)
df_rest


,name,lng,lat
0,언양휴게소,129.142144,35.597794
1,경주휴게소,129.192361,35.723736
2,김천휴게소,128.163945,36.130968
3,신탄진휴게소,127.418547,36.426745
4,천안삼거리휴게소,127.173357,36.788044
5,안성휴게소,127.132587,37.076674
6,영종대교휴게소,126.607049,37.553820


## 6. 각 휴게소를 경로상 가장 가까운 지점에 매칭 → 누적거리 계산


In [25]:
def nearest_cum_dist(route_coords, lng, lat):
    dists = np.hypot(route_coords["lng"] - lng, route_coords["lat"] - lat)
    idx = dists.idxmin()
    return route_coords.loc[idx, "cum_dist_km"], dists.loc[idx]

matched = []
for _, ra in df_rest.iterrows():
    cum_km, offset_deg = nearest_cum_dist(route_coords, ra["lng"], ra["lat"])
    matched.append({"휴게소": ra["name"], "누적거리_km": cum_km, "경로이탈도": offset_deg})

df_matched = pd.DataFrame(matched).sort_values("누적거리_km").reset_index(drop=True)
df_matched


,휴게소,누적거리_km,경로이탈도
0,언양휴게소,3.881129,0.584409
1,경주휴게소,68.789821,0.716875
2,김천휴게소,140.644320,0.104384
3,신탄진휴게소,241.278034,0.658151
4,천안삼거리휴게소,330.467092,0.463709
5,안성휴게소,334.652507,0.195893
6,영종대교휴게소,412.421547,0.125872


## 7. 구간(휴게소 ↔ 휴게소) 분할표 생성

이 표가 이후 ML 모델의 입력(X: 구간 거리, 구간 평균속도)으로 들어간다.


In [26]:
segments = []
prev_dist = 0.0
prev_name = ORIGIN_NAME

for _, row in df_matched.iterrows():
    seg_dist = row["누적거리_km"] - prev_dist
    segments.append({
        "출발": prev_name,
        "도착": row["휴게소"],
        "구간거리_km": round(seg_dist, 1),
        "누적거리_km": round(row["누적거리_km"], 1)
    })
    prev_dist = row["누적거리_km"]
    prev_name = row["휴게소"]

# 마지막 구간: 마지막 휴게소 -> 인천공항
total_km = df_roads["거리_m"].sum() / 1000
segments.append({
    "출발": prev_name,
    "도착": DESTINATION_NAME,
    "구간거리_km": round(total_km - prev_dist, 1),
    "누적거리_km": round(total_km, 1)
})

df_segments = pd.DataFrame(segments)
df_segments


,출발,도착,구간거리_km,누적거리_km
0,부산 물류센터,언양휴게소,3.9,3.9
1,언양휴게소,경주휴게소,64.9,68.8
2,경주휴게소,김천휴게소,71.9,140.6
3,김천휴게소,신탄진휴게소,100.6,241.3
4,신탄진휴게소,천안삼거리휴게소,89.2,330.5
5,천안삼거리휴게소,안성휴게소,4.2,334.7
6,안성휴게소,영종대교휴게소,77.8,412.4
7,영종대교휴게소,인천국제공항,11.0,423.4


## 8. 결과 저장

다음 노트북(ML 모델링)에서 이 파일을 불러와 각 구간에 대해 kWh/100km를 예측한다.


In [27]:
df_segments.to_csv("busan_incheon_route_segments.csv", index=False, encoding="utf-8-sig")
print("저장 완료: busan_incheon_route_segments.csv")


저장 완료: busan_incheon_route_segments.csv
